In [1]:
from pathlib import Path
import json, pickle
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

train = pd.read_parquet(ROOT / "data" / "processed" / "train.parquet")
with open(ROOT / "models" / "binning_process.pkl", "rb") as f:
    bp = pickle.load(f)
with open(ROOT / "models" / "selected_features.json") as f:
    final = json.load(f)

print(train.shape, len(final), "features")


(CVXPY) Nov 10 07:46:49 AM: Encountered unexpected exception importing solver HIGHS:
ImportError('/home/fiza/Credit_Risk/.venv/lib/python3.12/site-packages/highspy/_core.cpython-312-x86_64-linux-gnu.so: undefined symbol: _ZN5Highs13releaseMemoryEv')


(215257, 108) 16 features


In [2]:
all_features = list(bp.variable_names)
woe = bp.transform(train[all_features], metric="woe",
                   metric_missing="empirical", metric_special="empirical")
X = pd.DataFrame(np.asarray(woe), columns=all_features, index=train.index)[final]
y = train["TARGET"]
X.head()


,EXT_SOURCE_3,EXT_SOURCE_2,EXT_SOURCE_1,YEARS_EMPLOYED,BUREAU_DEBT_TO_CREDIT,AGE_YEARS,BUREAU_YEARS_SINCE_LAST,INST_LATE_SHARE_LAST12,BUREAU_YEARS_SINCE_FIRST,OCCUPATION_TYPE,ORGANIZATION_TYPE,AMT_CREDIT_TO_GOODS_RATIO,PREV_REFUSAL_RATE,NAME_INCOME_TYPE,BUREAU_ACTIVE_COUNT,PREV_GRANTED_TO_REQUESTED
0,-0.651492,0.015396,-0.057479,-0.167571,-0.574875,0.445490,-0.102815,0.193515,-0.044058,-0.296271,-0.419902,0.24596,0.142280,-0.190304,-0.034577,-0.284005
1,1.019244,0.789307,-0.057479,0.703441,0.056517,0.329503,0.417733,0.193515,0.199610,-0.296271,-0.036344,-0.30908,0.142280,0.075884,0.397718,0.125913
2,-0.148913,-0.166057,-0.057479,-0.350351,0.504085,-0.349363,0.417733,-0.334175,0.128031,0.271917,0.130652,0.24596,0.142280,-0.190304,0.146581,0.125913
3,-0.148913,0.232917,-0.057479,0.268827,0.551316,0.072052,0.417733,0.193515,0.231876,-0.205936,-0.153681,-0.52963,0.082381,0.075884,0.146581,-0.468482
4,1.019244,0.403710,-0.057479,0.147677,0.354242,0.445490,0.274338,0.193515,0.231876,0.236960,-0.262765,0.24596,0.142280,-0.190304,0.089042,0.125913


In [3]:
model = sm.Logit(y, sm.add_constant(X)).fit()
print(model.summary())


Optimization terminated successfully.
         Current function value: 0.247345
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                 TARGET   No. Observations:               215257
Model:                          Logit   Df Residuals:                   215240
Method:                           MLE   Df Model:                           16
Date:                Tue, 10 Nov 2026   Pseudo R-squ.:                  0.1183
Time:                        07:46:54   Log-Likelihood:                -53243.
converged:                       True   LL-Null:                       -60388.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                        -2.4365      0.009   -276.452      0.000      -2.

In [4]:
coef = model.params.drop("const").sort_values()
print(coef.round(3))
print("\nWrong sign (positive):", list(coef[coef > 0].index))
print("Not significant (p > 0.05):", list(model.pvalues.drop("const")[lambda p: p > 0.05].index))


EXT_SOURCE_2                -0.806
INST_LATE_SHARE_LAST12      -0.694
AMT_CREDIT_TO_GOODS_RATIO   -0.677
EXT_SOURCE_3                -0.654
BUREAU_DEBT_TO_CREDIT       -0.608
EXT_SOURCE_1                -0.568
PREV_GRANTED_TO_REQUESTED   -0.486
PREV_REFUSAL_RATE           -0.451
BUREAU_ACTIVE_COUNT         -0.427
ORGANIZATION_TYPE           -0.423
YEARS_EMPLOYED              -0.374
OCCUPATION_TYPE             -0.362
BUREAU_YEARS_SINCE_FIRST    -0.221
NAME_INCOME_TYPE            -0.070
BUREAU_YEARS_SINCE_LAST      0.000
AGE_YEARS                    0.174
dtype: float64

Wrong sign (positive): ['BUREAU_YEARS_SINCE_LAST', 'AGE_YEARS']
Not significant (p > 0.05): ['BUREAU_YEARS_SINCE_LAST', 'NAME_INCOME_TYPE']


In [5]:
pd_train = model.predict(sm.add_constant(X))
auc = roc_auc_score(y, pd_train)
print(f"AUC  = {auc:.3f}")
print(f"Gini = {2*auc - 1:.3f}")


AUC  = 0.755
Gini = 0.511


In [6]:
features_lr = list(final)

while True:
    model = sm.Logit(y, sm.add_constant(X[features_lr])).fit(disp=0)
    coef = model.params.drop("const")
    pval = model.pvalues.drop("const")

    bad = pval[(coef > 0) | (pval > 0.05)]      # wrong sign OR not significant
    if bad.empty:
        break                                   # all clean → stop

    worst = bad.idxmax()
    print(f"drop {worst:28s} coef = {coef[worst]:+.3f}   p = {pval[worst]:.3f}")
    features_lr.remove(worst)

auc = roc_auc_score(y, model.predict(sm.add_constant(X[features_lr])))
print(f"\n{len(features_lr)} features kept, Gini = {2*auc - 1:.3f}")
model.params.round(3)


drop BUREAU_YEARS_SINCE_LAST      coef = +0.000   p = 0.995
drop NAME_INCOME_TYPE             coef = -0.070   p = 0.129
drop AGE_YEARS                    coef = +0.163   p = 0.000

13 features kept, Gini = 0.511


const                       -2.436
EXT_SOURCE_3                -0.649
EXT_SOURCE_2                -0.802
EXT_SOURCE_1                -0.541
YEARS_EMPLOYED              -0.325
BUREAU_DEBT_TO_CREDIT       -0.604
INST_LATE_SHARE_LAST12      -0.691
BUREAU_YEARS_SINCE_FIRST    -0.204
OCCUPATION_TYPE             -0.375
ORGANIZATION_TYPE           -0.411
AMT_CREDIT_TO_GOODS_RATIO   -0.680
PREV_REFUSAL_RATE           -0.451
BUREAU_ACTIVE_COUNT         -0.429
PREV_GRANTED_TO_REQUESTED   -0.494
dtype: float64